In [7]:
import os
from tensorboard.backend.event_processing import event_accumulator
import pandas as pd

# Корневая директория с логами TensorBoard
root_dir = "nsgaii_training_logs"  # Путь из скриншота

# Проверка существования корневой директории
if not os.path.exists(root_dir):
    print(f"Ошибка: Директория {root_dir} не существует. Проверьте путь.")
    exit()

# Инициализация общего словаря и списка DataFrame
all_data = {}
all_dfs = []

# Обработка каждой поддиректории
for subdir in os.listdir(root_dir):
    subdir_path = os.path.join(root_dir, subdir)
    if not os.path.isdir(subdir_path):
        continue

    print(f"\nОбработка поддиректории: {subdir}")

    # Проверка наличия файлов логов
    log_files = [f for f in os.listdir(subdir_path) if f.startswith("events.out.tfevents")]
    if not log_files:
        print(f"В поддиректории {subdir} не найдены файлы логов TensorBoard (events.out.tfevents.*).")
        continue

    print(f"Найдено {len(log_files)} файлов логов: {log_files}")

    # Обработка каждого файла логов в поддиректории
    subdir_data = {}
    for log_file in log_files:
        log_path = os.path.join(subdir_path, log_file)
        print(f"  Обработка файла: {log_file}")

        # Инициализация EventAccumulator
        ea = event_accumulator.EventAccumulator(log_path)
        try:
            ea.Reload()
        except Exception as e:
            print(f"  Ошибка при загрузке файла {log_file}: {str(e)}")
            continue

        # Получение всех scalar-тегов
        tags = ea.Tags()['scalars']
        if not tags:
            print(f"  В файле {log_file} не найдены скалярные метрики. Доступные типы данных: {ea.Tags()}")
            continue

        print(f"  Найдены теги: {tags}")

        # Извлечение данных для каждого тега
        for tag in tags:
            events = ea.Scalars(tag)
            if not events:
                print(f"  Тег {tag} не содержит данных.")
                continue
            if tag not in subdir_data:
                subdir_data[tag] = []
            for event in events:
                subdir_data[tag].append({
                    'step': event.step,
                    'value': event.value,
                    'wall_time': event.wall_time,
                    'subdir': subdir  # Добавляем имя поддиректории для идентификации
                })

    # Добавление данных поддиректории в общий словарь
    all_data.update(subdir_data)

    # Преобразование данных поддиректории в DataFrame
    for tag in subdir_data:
        df = pd.DataFrame(subdir_data[tag])
        df['tag'] = tag
        all_dfs.append(df)

# Проверка, есть ли данные
if not all_data:
    print("Ошибка: Словарь данных пуст. Проверьте наличие скалярных метрик в логах.")
    exit()

# Объединяем все DataFrame в один
combined_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else None

# Вывод словаря
print("\nДанные в виде словаря:")
for tag, values in all_data.items():
    print(f"\nТег: {tag}")
    for value in values:
        print(f"Step: {value['step']}, Value: {value['value']}, Wall Time: {value['wall_time']}, Subdir: {value['subdir']}")

# Вывод DataFrame
if combined_df is not None:
    print("\nДанные в виде DataFrame:")
    print(combined_df)

    # Сохранение в CSV
    combined_df.to_csv("tensorboard_scalars_all.csv", index=False)
    print("\nДанные сохранены в tensorboard_scalars_all.csv")
else:
    print("DataFrame не создан, так как нет данных.")


Обработка поддиректории: neural_bco_optimization_exp_weighted_connectivity_vologda_pp_0.5_op_0_cp_0.5_generated
Найдено 5 файлов логов: ['events.out.tfevents.1760019428.gk.226071.1', 'events.out.tfevents.1760018972.gk.226071.0', 'events.out.tfevents.1760020805.gk.248228.0', 'events.out.tfevents.1760019856.gk.226071.3', 'events.out.tfevents.1760019480.gk.226071.2']
  Обработка файла: events.out.tfevents.1760019428.gk.226071.1
  В файле events.out.tfevents.1760019428.gk.226071.1 не найдены скалярные метрики. Доступные типы данных: {'images': [], 'audio': [], 'histograms': [], 'scalars': [], 'distributions': [], 'tensors': ['device type/text_summary', 'random seed/text_summary', 'n_bees/text_summary', 'n_iterations/text_summary', 'batch_size/text_summary', 'neural_bees/text_summary', 'force_linking_unlinked/text_summary', 'experiment.logdir/text_summary', 'experiment.anomaly/text_summary', 'experiment.cpu/text_summary', 'experiment.seed/text_summary', 'experiment.symmetric_routes/text_su

In [1]:
!pip show blocksnet

In [2]:
!pip install blocksnet

  Using cached blocksnet-0.1.0-py3-none-any.whl.metadata (13 kB)
  Using cached geopandas-0.14.4-py3-none-any.whl.metadata (1.5 kB)
  Using cached osmnx-1.9.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached pandera-0.20.2-py3-none-any.whl.metadata (15 kB)
  Using cached PuLP-2.7.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached voronoi_diagram_for_polygons-0.1.6-py3-none-any.whl.metadata (8.9 kB)
  Using cached momepy-0.7.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached iduedu-0.1.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached osm2geojson-0.2.9-py3-none-any.whl.metadata (4.2 kB)
  Using cached multimethod-1.10-py3-none-any.whl.metadata (8.2 kB)
  Using cached typeguard-4.4.4-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached networkx-3.3-py3-none-any.whl.metadata (5.1 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadat

In [8]:
combined_df

,step,value,wall_time,subdir,tag
0,0,1.318288,1.760021e+09,neural_bco_optimization_exp_weighted_connectiv...,best cost
1,1,1.260402,1.760022e+09,neural_bco_optimization_exp_weighted_connectiv...,best cost
2,2,1.244692,1.760022e+09,neural_bco_optimization_exp_weighted_connectiv...,best cost
3,3,1.237294,1.760023e+09,neural_bco_optimization_exp_weighted_connectiv...,best cost
4,4,1.186333,1.760024e+09,neural_bco_optimization_exp_weighted_connectiv...,best cost
...,...,...,...,...,...
202995,1998,80.046059,1.761119e+09,nsgaii_exp_weighted_connectivity_mandl_pp_0.33...,best median_connectivity_weighted
202996,1999,80.046059,1.761119e+09,nsgaii_exp_weighted_connectivity_mandl_pp_0.33...,best median_connectivity_weighted
202997,2000,80.046059,1.761119e+09,nsgaii_exp_weighted_connectivity_mandl_pp_0.33...,best median_connectivity_weighted
202998,0,80.046059,1.760982e+09,nsgaii_exp_weighted_connectivity_mandl_pp_0.33...,best median_connectivity_weighted


In [16]:
import os
from tensorboard.backend.event_processing import event_accumulator
import pandas as pd

# Корневая директория с логами TensorBoard
root_dir = "nsgaii_training_logs"

# Проверка существования корневой директории
if not os.path.exists(root_dir):
    print(f"Ошибка: Директория {root_dir} не существует. Проверьте путь.")
    exit()

# Список всех скалярных метрик (только с префиксом best)
all_scalars = [
    "best # disconnected node pairs",
    "best # stops out of bounds",
    "best $d_0$",
    "best $d_1$",
    "best $d_2$",
    "best $d_{un}$",
    "best ATT",
    "best RTT",
    "best cost",
    "best median_connectivity",
    "best median_connectivity_weighted"
]

# Ключевые параметры для поиска минимальных значений
key_scalars = [
    "best median_connectivity_weighted",
    "best ATT",
    "best RTT"
]

# Словарь для хранения всех Парето-фронтов
all_pareto_fronts = {}

# Обработка каждой поддиректории
for subdir in os.listdir(root_dir):
    subdir_path = os.path.join(root_dir, subdir)
    if not os.path.isdir(subdir_path):
        continue

    print(f"\n=== Обработка эксперимента: {subdir} ===")

    # Проверка наличия файлов логов
    log_files = [f for f in os.listdir(subdir_path) if f.startswith("events.out.tfevents")]
    if not log_files:
        print(f"В поддиректории {subdir} не найдены файлы логов TensorBoard (events.out.tfevents.*).")
        continue

    print(f"Найдено {len(log_files)} файлов логов: {log_files}")

    # Инициализация словаря для хранения данных текущей поддиректории
    subdir_data = {scalar: [] for scalar in all_scalars}

    # Обработка каждого файла логов
    for log_file in log_files:
        log_path = os.path.join(subdir_path, log_file)
        print(f"  Обработка файла: {log_file}")

        # Инициализация EventAccumulator
        ea = event_accumulator.EventAccumulator(log_path)
        try:
            ea.Reload()
        except Exception as e:
            print(f"  Ошибка при загрузке файла {log_file}: {str(e)}")
            continue

        # Проверка наличия скалярных метрик
        tags = ea.Tags().get('scalars', [])
        if not tags:
            print(f"  В файле {log_file} не найдены скалярные метрики. Доступные типы данных: {ea.Tags()}")
            continue

        print(f"  Доступные теги: {tags}")

        # Извлечение данных только для тегов с префиксом best
        for scalar in all_scalars:
            if scalar in tags:
                events = ea.Scalars(scalar)
                for event in events:
                    subdir_data[scalar].append({
                        'step': event.step,
                        'value': event.value,
                        'wall_time': event.wall_time
                    })

    # Преобразование данных в DataFrame
    dfs = []
    for scalar in all_scalars:
        if subdir_data[scalar]:
            df = pd.DataFrame(subdir_data[scalar])
            df['tag'] = scalar
            dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True) if dfs else None

    # Проверка наличия данных
    if combined_df is None or combined_df.empty:
        print(f"Ошибка: Нет данных для обработки в поддиректории {subdir}.")
        continue

    # Нахождение минимальных значений для ключевых параметров и соответствующих шагов
    min_steps = {}
    for key_scalar in key_scalars:
        if key_scalar in combined_df['tag'].unique():
            min_value = combined_df[combined_df['tag'] == key_scalar]['value'].min()
            min_step = combined_df[(combined_df['tag'] == key_scalar) & (combined_df['value'] == min_value)]['step'].iloc[0]
            min_steps[key_scalar] = min_step
            print(f"Минимальное значение для {key_scalar}: {min_value} на шаге {min_step}")

    # Формирование DataFrame с Парето-фронтом
    if min_steps:
        pareto_data = []
        for key_scalar, min_step in min_steps.items():
            row = {'experiment': subdir, 'key_scalar': key_scalar, 'min_step': min_step}
            for scalar in all_scalars:
                value = combined_df[(combined_df['tag'] == scalar) & (combined_df['step'] == min_step)]['value'].iloc[0] if not combined_df[(combined_df['tag'] == scalar) & (combined_df['step'] == min_step)].empty else None
                row[scalar] = value
            pareto_data.append(row)

        # Создание DataFrame и сохранение в общий словарь
        pareto_df = pd.DataFrame(pareto_data)
        all_pareto_fronts[subdir] = pareto_df
        print(f"\nПарето-фронт для эксперимента {subdir} (DataFrame):")
        print(pareto_df)
    else:
        print(f"В поддиректории {subdir} не найдены минимальные значения для ключевых параметров.")

# Вывод итогового словаря с всеми Парето-фронтами
print("\n=== Итоговый словарь со всеми Парето-фронтами ===")
for subdir, df in all_pareto_fronts.items():
    print(f"\nЭксперимент: {subdir}")
    print(df)


=== Обработка эксперимента: neural_bco_optimization_exp_weighted_connectivity_vologda_pp_0.5_op_0_cp_0.5_generated ===
Найдено 5 файлов логов: ['events.out.tfevents.1760019428.gk.226071.1', 'events.out.tfevents.1760018972.gk.226071.0', 'events.out.tfevents.1760020805.gk.248228.0', 'events.out.tfevents.1760019856.gk.226071.3', 'events.out.tfevents.1760019480.gk.226071.2']
  Обработка файла: events.out.tfevents.1760019428.gk.226071.1
  В файле events.out.tfevents.1760019428.gk.226071.1 не найдены скалярные метрики. Доступные типы данных: {'images': [], 'audio': [], 'histograms': [], 'scalars': [], 'distributions': [], 'tensors': ['device type/text_summary', 'random seed/text_summary', 'n_bees/text_summary', 'n_iterations/text_summary', 'batch_size/text_summary', 'neural_bees/text_summary', 'force_linking_unlinked/text_summary', 'experiment.logdir/text_summary', 'experiment.anomaly/text_summary', 'experiment.cpu/text_summary', 'experiment.seed/text_summary', 'experiment.symmetric_routes/

In [24]:


# Вариант 1: Использование словарного включения
filtered_dict = {key: value for key, value in all_pareto_fronts.items() if key.startswith("nsgaii")}

# Вариант 2: Использование filter() и lambda
filtered_dict_alt = dict(filter(lambda item: item[0].startswith("nsgaii"), all_pareto_fronts.items()))

print(filtered_dict.keys())
# Вывод: {'nsgaii_param1': 10, 'nsgaii_param2': 20, 'nsgaii_result': 40}

dict_keys(['nsgaii_exp_weighted_connectivity_mumford1_pp_0.33_op_0.33_cp_0.33_starting', 'nsgaii_exp_weighted_connectivity_mumford2_pp_0.33_op_0.33_cp_0.33_starting', 'nsgaii_exp_weighted_connectivity_mumford3_pp_0.33_op_0.33_cp_0.33_starting', 'nsgaii_exp_weighted_connectivity_mumford0_pp_0.33_op_0.33_cp_0.33_starting', 'nsgaii_exp_weighted_connectivity_mandl_pp_0.33_op_0.33_cp_0.33_starting'])


In [21]:
list(filtered_dict.keys())

['nsgaii_exp_weighted_connectivity_mumford1_pp_0.33_op_0.33_cp_0.33_starting',
 'nsgaii_exp_weighted_connectivity_mumford2_pp_0.33_op_0.33_cp_0.33_starting',
 'nsgaii_exp_weighted_connectivity_mumford3_pp_0.33_op_0.33_cp_0.33_starting',
 'nsgaii_exp_weighted_connectivity_mumford0_pp_0.33_op_0.33_cp_0.33_starting',
 'nsgaii_exp_weighted_connectivity_mandl_pp_0.33_op_0.33_cp_0.33_starting']

In [23]:
filtered_dict['nsgaii_exp_weighted_connectivity_mumford0_pp_0.33_op_0.33_cp_0.33_starting']

,experiment,key_scalar,min_step,best # disconnected node pairs,best # stops out of bounds,best $d_0$,best $d_1$,best $d_2$,best $d_{un}$,best ATT,best RTT,best cost,best median_connectivity,best median_connectivity_weighted
0,nsgaii_exp_weighted_connectivity_mumford0_pp_0...,best median_connectivity_weighted,49,0.0,0.0,16.515665,23.611759,0.0,0.0,15.338438,218.0,0.217587,808.0,338.906219
1,nsgaii_exp_weighted_connectivity_mumford0_pp_0...,best ATT,1682,0.0,0.0,16.720247,23.974163,0.0,0.0,15.020750,198.0,0.217248,808.0,338.906219
2,nsgaii_exp_weighted_connectivity_mumford0_pp_0...,best RTT,617,0.0,0.0,16.720247,23.991699,0.0,0.0,15.025455,198.0,0.217248,808.0,338.906219


In [15]:
for e in pareto_df.experiment:
    print(e)

nsgaii_exp_weighted_connectivity_mandl_pp_0.33_op_0.33_cp_0.33_starting
nsgaii_exp_weighted_connectivity_mandl_pp_0.33_op_0.33_cp_0.33_starting
nsgaii_exp_weighted_connectivity_mandl_pp_0.33_op_0.33_cp_0.33_starting


In [29]:
import os
import re
import pandas as pd
import numpy as np
from tensorboard.backend.event_processing import event_accumulator

# === Настройки ===
root_dir = "nsgaii_training_logs"
output_file = "tensorboard_weighted_results.csv"

# --- Количество маршрутов в каждом датасете ---
n_routes = {
    "mandl": 6,
    "mumford0": 12,
    "mumford1": 15,
    "mumford2": 56,
    "mumford3": 60
}

# --- Метрики для извлечения ---
all_scalars = [
    "best ATT",
    "best RTT",
    "best median_connectivity",
    "best median_connectivity_weighted",
    "best cost",
    "best $d_{un}$",
    "best $d_0$",
    "best $d_1$",
    "best $d_2$"
]

key_scalars = [
    "best ATT",
    "best RTT",
    "best median_connectivity_weighted"
]

# --- Комбинации весов ---
weight_combinations = [
    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
    (0.5, 0.5, 0),
    (0.5, 0, 0.5),
    (0, 0.5, 0.5),
    (0.33, 0.33, 0.33),
]

# === Основной цикл ===
results = []

for subdir in sorted(os.listdir(root_dir)):
    subdir_path = os.path.join(root_dir, subdir)
    if not os.path.isdir(subdir_path):
        continue

    # Извлекаем название датасета
    match = re.search(r"(mandl|mumford\d*)", subdir)
    if not match:
        continue
    dataset_name = match.group(1)

    # Поиск TensorBoard логов
    log_files = [f for f in os.listdir(subdir_path) if f.startswith("events.out.tfevents")]
    if not log_files:
        continue

    # Чтение логов
    subdir_data = {scalar: [] for scalar in all_scalars}
    for log_file in log_files:
        log_path = os.path.join(subdir_path, log_file)
        ea = event_accumulator.EventAccumulator(log_path)
        try:
            ea.Reload()
        except Exception:
            continue

        tags = ea.Tags().get("scalars", [])
        for scalar in all_scalars:
            if scalar in tags:
                for event in ea.Scalars(scalar):
                    subdir_data[scalar].append({"step": event.step, "value": event.value})

    dfs = []
    for scalar, values in subdir_data.items():
        if values:
            df = pd.DataFrame(values)
            df["tag"] = scalar
            dfs.append(df)
    if not dfs:
        continue

    combined_df = pd.concat(dfs, ignore_index=True)
    pivot_df = combined_df.pivot_table(index="step", columns="tag", values="value", aggfunc="last").reset_index()

    if not all(k in pivot_df.columns for k in key_scalars):
        continue

    # --- Нормализация RTT ---
    if dataset_name in n_routes:
        pivot_df["best RTT norm"] = pivot_df["best RTT"] / n_routes[dataset_name]

    # --- Перебор весов ---
    for (dt, rt, ct) in weight_combinations:
        pivot_df["weighted_sum"] = (
            dt * pivot_df["best ATT"] +
            rt * pivot_df["best RTT norm"] +
            ct * pivot_df["best median_connectivity_weighted"]
        )

        best_idx = pivot_df["weighted_sum"].idxmin()
        best_row = pivot_df.loc[best_idx]

        row = {
            "dataset": dataset_name,
            "demand_time": dt,
            "route_time": rt,
            "connectivity": ct,
            "ATT": best_row.get("best ATT", np.nan),
            "RTT": best_row.get("best RTT", np.nan),
            "median_connectivity": best_row.get("best median_connectivity", np.nan),
            "median_connectivity_weighted": best_row.get("best median_connectivity_weighted", np.nan),
            "cost": best_row.get("best cost", np.nan),
            "$d_{un}$": best_row.get("best $d_{un}$", np.nan),
            "$d_0$": best_row.get("best $d_0$", np.nan),
            "$d_1$": best_row.get("best $d_1$", np.nan),
            "$d_2$": best_row.get("best $d_2$", np.nan),
        }
        results.append(row)

# === Формирование итоговой таблицы ===
columns_order = [
    "dataset", "demand_time", "route_time", "connectivity",
    "ATT", "RTT", "median_connectivity", "median_connectivity_weighted",
    "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
]

df = pd.DataFrame(results)[columns_order]

# === Порядок датасетов и весов ===
dataset_order = ["mandl", "mumford0", "mumford1", "mumford2", "mumford3"]
df["dataset"] = pd.Categorical(df["dataset"], categories=dataset_order, ordered=True)
df = df.sort_values(["dataset", "demand_time", "route_time", "connectivity"], ignore_index=True)

# === Форматирование чисел ===
metric_cols = [
    "ATT", "RTT", "median_connectivity", "median_connectivity_weighted",
    "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
]
df[metric_cols] = df[metric_cols].applymap(lambda x: round(float(x), 4) if pd.notna(x) else x)

# === Сохранение CSV ===
df.to_csv(output_file, index=False)
print(f"\n✅ Результаты сохранены в '{output_file}' (RTT делится на n_routes).")



✅ Результаты сохранены в 'tensorboard_weighted_results.csv' (RTT делится на n_routes).


/tmp/ipykernel_367629/2099671942.py:151: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[metric_cols] = df[metric_cols].applymap(lambda x: round(float(x), 4) if pd.notna(x) else x)
